In [ ]:
print("all ok")

all ok


Install Dependencies

In [ ]:
!pip -q install unsloth
!pip -q install transformers==4.56.2
!pip -q install --no-deps trl==0.22.2
!pip -q install datasets huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 125.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 112.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3

Hugging Face Login

In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

print("✅ Hugging Face Authenticated")

✅ Hugging Face Authenticated


Imports & Configuration

In [ ]:
import os
import gc
import torch

from datasets import load_dataset

from unsloth import FastLanguageModel

from trl import (
    SFTTrainer,
    SFTConfig,
)

from huggingface_hub import (
    create_repo,
    upload_folder,
)

MAX_SEQ_LENGTH = 2048

STAGE1_MODEL = (
    "Pzazz55/insurance-ai-assistant-finetuning-stage1"
)

STAGE2_MODEL = (
    "Pzazz55/insurance-ai-assistant-finetuning-stage2"
)

os.makedirs("/content/models", exist_ok=True)
os.makedirs("/content/reports", exist_ok=True)
os.makedirs("/content/outputs_sft", exist_ok=True)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Create Stage 2 HF Repository

In [ ]:
create_repo(
    repo_id=STAGE2_MODEL,
    exist_ok=True,
)

print("✅ Stage 2 Repository Ready")

✅ Stage 2 Repository Ready


Load Stage 1 Model From HF

In [ ]:
print("Loading Stage 1 model...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=STAGE1_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

print("✅ Stage 1 Model Loaded")

Loading Stage 1 model...
==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/499 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

✅ Stage 1 Model Loaded


Apply Additional LoRA For SFT

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print("✅ SFT LoRA Attached")

Unsloth 2026.6.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ SFT LoRA Attached


Alpaca Prompt Template

In [ ]:
ALPACA_PROMPT = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
{response}"""

Load Instruction Dataset

In [ ]:
from datasets import load_dataset

# Load only the specific .jsonl file directly using your HF Repo ID
dataset = load_dataset(
    "Pzazz55/insurance-ai-assistant-finetuning-dataset",
    data_files="insurance_instruction_dataset.jsonl"
)

# Access your data inside the 'train' split
print(f"✅ Successfully loaded specific file!")
print(f"Total records: {len(dataset['train'])}")

insurance_instruction_dataset.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

✅ Successfully loaded specific file!
Total records: 120


Format Dataset

In [ ]:
def format_examples(examples):

    formatted_texts = []

    for instruction, response in zip(
        examples["instruction"],
        examples["output"]
    ):
        formatted_texts.append(
            ALPACA_PROMPT.format(
                instruction=instruction.strip(),
                response=response.strip()
            )
        )

    return {"text": formatted_texts}


dataset = dataset.map(
    format_examples,
    batched=True,
    desc="Formatting dataset using Alpaca template"
)

print("✅ Alpaca formatting applied")

Formatting dataset using Alpaca template:   0%|          | 0/120 [00:00<?, ? examples/s]

✅ Alpaca formatting applied


Validate Dataset

In [ ]:
print(dataset["train"][0]["text"])

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Explain auto insurance in insurance context.

### Response:
Auto insurance covers financial losses from vehicle accidents, theft, and damage. It typically includes liability, collision, and comprehensive coverage. It plays a key role in insurance operations, including policy administration, claims processing, risk assessment, and customer servicing. Understanding this concept helps policyholders and insurers make informed decisions.


Configure SFT

In [ ]:
sft_config = SFTConfig(
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=100,
    learning_rate=2e-5,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    output_dir="outputs_sft",
    report_to="none",
    save_strategy="no",
)

Create Trainer

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    args=sft_config,
)

print("✅ SFT Trainer Initialized")
print(f"Training Records: {len(dataset['train'])}")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/120 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=6):   0%|          | 0/120 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!
✅ SFT Trainer Initialized
Training Records: 120


Train

In [ ]:
trainer.train()
print("✅ Stage 2 Training Complete")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6 | Num Epochs = 50 | Total steps = 100
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,3.335600
10,3.089200
15,2.584200
20,2.146400
25,1.793800
30,1.509700
35,1.283800
40,1.097600
45,0.929000
50,0.781500


✅ Stage 2 Training Complete


Save Stage 2

In [ ]:
stage2_path = (
    "/content/models/stage2_sft"
)

model.save_pretrained_merged(
    stage2_path,
    tokenizer,
    save_method="merged_16bit",
)

print("✅ Stage 2 Saved")

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/content/models/stage2_sft`: 100%|██████████| 1/1 [00:59<00:00, 59.77s/it]


Successfully copied all 1 files from cache to `/content/models/stage2_sft`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:33<00:00, 93.53s/it]


Unsloth: Merge process complete. Saved to `/content/models/stage2_sft`
✅ Stage 2 Saved


Upload Stage 2 To HF

In [ ]:
upload_folder(
    repo_id=STAGE2_MODEL,
    folder_path=stage2_path,
)

print("✅ Stage 2 Uploaded")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...stage2_sft/tokenizer.json:  97%|#########7| 11.1MB / 11.4MB            

  ...ge2_sft/model.safetensors:   1%|1         | 40.0MB / 3.09GB            

✅ Stage 2 Uploaded


Reload Stage 2 From HF

In [ ]:
del trainer
del model

gc.collect()

torch.cuda.empty_cache()

model, tokenizer = (
    FastLanguageModel.from_pretrained(
        model_name=STAGE2_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
)

FastLanguageModel.for_inference(model)

print("✅ Stage 2 Reloaded From HF")

==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

✅ Stage 2 Reloaded From HF


Evaluation Questions

In [ ]:
evaluation_questions = [
    "What mandatory items must be present in a Middle Market risk submission before it can be processed for binding?",
    "How does a prior history of lapses in commercial auto coverage impact premium calculations and overall risk acceptability?",
    "What specific criteria determine if a commercial property qualifies for highly protected risk (HPR) status?",
    "Explain how aggregate limit extensions operate within a commercial general liability policy during a catastrophic loss year.",
    "What underwriting indicators necessitate the inclusion of a specialized environmental pollution liability rider?",
    "How should an underwriter evaluate the risk profile of a manufacturing plant utilizing aging machinery without telematics tracking?",
    "What are the core differences between a claims-made policy form and an occurrence policy form regarding professional liability?",
    "Under what specific conditions is a retroactive date adjustment permitted on an executive directors and officers (D&O) liability policy?",
    "How does a business interruption policy handle contingent business income losses if a key downstream supplier suffers a fire?",
    "What risk mitigation factors can offset a high experience modification rate (E-Mod) when underwriting worker's compensation?"
]

Alpaca Inference Function

In [ ]:
def generate_answer(question):

    prompt = f"""### Instruction:
{question}

### Response:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            temperature=0.1,
        )

    text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True,
    )

    answer = text.split(
        "### Response:"
    )[-1].strip()

    return answer

Comparison Data

In [ ]:
comparison_data = [
    {
        "question":
        "What mandatory items must be present in a Middle Market risk submission before it can be processed for binding?",

        "base_answer":
        "Provides generic business information requirements.",

        "reason":
        "Stage 2 should provide insurance-specific submission requirements."
    },

    # Add remaining 9 records
]

Generate Comparison Report

In [ ]:
# # Generate Report Structure
# markdown = [
#     "# Stage 1 Evaluation Report",
#     "",
#     "| Question | Fine-Tuned Model Answer |",
#     "|---|---|",
# ]

# print("🚀 Starting Evaluation Loop...")
# print("-" * 50)

# # Loop cleanly through all 10 custom questions
# for idx, question in enumerate(evaluation_questions, 1):
#     # 1. Print the current progress and question to console
#     print(f"\n[Question {idx}/{len(evaluation_questions)}]: {question}")
#     print("-" * 50)

#     # 2. Generate the model's prediction
#     answer = generate_answer(question)

#     # 3. Print the answer directly to the console
#     print(f"[Model Answer]:\n{answer}")
#     print("-" * 50)

#     # 4. Append to the markdown file logic (preserving the HTML line breaks for table formatting)
#     markdown.append(
#         f"| {question} | {answer.replace(chr(10), '<br>')} |"
#     )

# # Save the final report
# with open("/content/reports/stage2_evaluation_report.md", "w", encoding="utf-8") as f:
#     f.write("\n".join(markdown))

# print("\n✅ Evaluation complete! Report saved to /content/reports/stage2_evaluation_report.md")

In [ ]:
# import os
# import re
# import torch
# from unsloth import FastLanguageModel
# from safetensors.torch import load_file
# from huggingface_hub import HfApi

# # =====================================================================
# # CONFIGURATION & SEPARATE REPOSITORY SETUP
# # =====================================================================
# max_seq_length = 2048
# production_model_path = "/content/models/stage2_sft"

# # Separate input/output configs aligned with your update
# INPUT_REPO_ID = "Pzazz55/insurance-ai-assistant-finetuning-stage1"
# OUTPUT_REPO_ID = "Pzazz55/insurance-ai-assistant-finetuning-stage2"
# # Ensure HF_TOKEN is defined globally or in your environment variables
# # HF_TOKEN = "YOUR_HF_WRITE_TOKEN"

# local_dir = "/content/reports"
# os.makedirs(local_dir, exist_ok=True)

# input_report_path = os.path.join(local_dir, "stage1_evaluation_report.md")
# output_report_path = os.path.join(local_dir, "stage2_evaluation_report.md")

# if not os.path.exists(production_model_path):
#     raise FileNotFoundError(f"Could not find the model path at {production_model_path}. Ensure Stage 2 SFT completed successfully.")

# if not os.path.exists(input_report_path):
#     raise FileNotFoundError(
#         f"Could not find {input_report_path}. Please make sure Stage 1 script ran and generated it first!"
#     )

# # =====================================================================
# # 1. LOCAL PRODUCTION MODEL INITIALIZATION (DIRECT SAFETENSORS MAPPING)
# # =====================================================================
# print("🧠 Loading underlying base model architecture blueprint...")
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = "unsloth/Qwen2.5-1.5B-Instruct",
#     max_seq_length = max_seq_length,
#     load_in_4bit = True,
# )

# print(f"🧬 Injecting custom Stage 2 SFT Safetensors weights from {production_model_path}...")
# safetensors_weight_path = os.path.join(production_model_path, "model.safetensors")
# state_dict = load_file(safetensors_weight_path, device="cuda")

# # Map local fine-tuned weights matrix onto active architecture structural skeleton
# model.load_state_dict(state_dict, strict=False)
# FastLanguageModel.for_inference(model)
# print("✅ SFT Production Model loaded successfully and ready for evaluation!\n")

# # =====================================================================
# # 2. READ AND PARSE THE EXISTING STAGE 1 MARKDOWN REPORT
# # =====================================================================
# print(f"📖 Reading existing report from {input_report_path}...")
# with open(input_report_path, "r", encoding="utf-8") as f:
#     lines = f.readlines()

# markdown = []
# existing_rows = []

# for line in lines:
#     line_str = line.strip()

#     # Update title header alignment
#     if line_str.startswith("# Stage 1 Evaluation Report"):
#         markdown.append("# Stage 2 Evaluation Report")
#     # Update the layout headers to add the fine-tuned evaluation dimension
#     elif "| Question |" in line_str and "Model Answer" in line_str:
#         markdown.append("| Question | Base Model Answer | Fine-Tuned Model Answer |")
#     # Balance table margins gracefully
#     elif any(sep in line_str for sep in ["|---|---|", "|---|---|---|", "| :--- | :--- |"]):
#         markdown.append("| :--- | :--- | :--- |")
#     # Parse existing evaluation table rows
#     elif line_str.startswith("|") and not line_str.startswith("| Question"):
#         parts = [p.strip() for p in line_str.split("|")[1:-1]]
#         if len(parts) >= 2:
#             question = parts[0]
#             base_answer = parts[1]
#             existing_rows.append((question, base_answer))
#     elif line_str != "":
#         markdown.append(line_str)

# if markdown[-1] != "":
#     markdown.append("")

# print(f"📋 Found {len(existing_rows)} rows to update.")
# print("🚀 Starting Fine-Tuned Evaluation Loop using Token Slicing...")
# print("-" * 50)

# # =====================================================================
# # 3. EVALUATION MATRIX GENERATION LOOP
# # =====================================================================
# for idx, (question, base_answer) in enumerate(existing_rows, 1):
#     # Convert cell breaks back into plaintext formatting strings for inference execution
#     clean_question = question.replace("<br>", "\n").replace("<br/>", "\n")

#     print(f"\n[Question {idx}/{len(existing_rows)}]: {clean_question}")
#     print("-" * 50)

#     # Structure text inside aligned ChatML structure using chat templates
#     messages = [
#         {"role": "system", "content": "You are an expert Insurance Underwriting Assistant. Provide clear and legally accurate domain responses."},
#         {"role": "user", "content": clean_question}
#     ]
#     formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#     inputs = tokenizer([formatted_prompt], return_tensors = "pt").to("cuda")

#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens = 300,
#             use_cache = True,
#             eos_token_id = tokenizer.eos_token_id,
#             temperature = 0.1
#         )

#     # Slice output sequence to pull exactly the freshly minted inference tokens
#     input_len = inputs.input_ids.shape[1]
#     generated_tokens = outputs[0][input_len:]
#     ft_answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

#     print(f"[Fine-Tuned Model Answer]:\n{ft_answer}")
#     print("-" * 50)

#     # Sanitize markdown strings against formatting breaks
#     md_safe_ft_answer = ft_answer.replace("\n", "<br>").replace("|", "\\|")

#     # Append the completed data string straight back to the dataset row stack
#     markdown.append(f"| {question} | {base_answer} | {md_safe_ft_answer} |")

# # =====================================================================
# # 4. EXPORT COMPILED MARKDOWN REPORT TO LOCAL STORAGE
# # =====================================================================
# with open(output_report_path, "w", encoding="utf-8") as f:
#     f.write("\n".join(markdown))

# print(f"\n✅ Evaluation complete! Local stage 2 report saved to {output_report_path}")
# print("-" * 50)

# # =====================================================================
# # 5. STREAM REMOTE TRANSFER DIRECT TO STAGE 2 HUGGING FACE RESULTS FOLDER
# # =====================================================================
# print(f"📤 Uploading report to Hugging Face Hub under Stage 2 repository results folder...")
# try:
#     api = HfApi(token=HF_TOKEN)
#     path_in_repo = "results/stage2_evaluation_report.md"

#     api.upload_file(
#         path_or_fileobj=output_report_path,
#         path_in_repo=path_in_repo,
#         repo_id=OUTPUT_REPO_ID,  # FIXED: Target the Stage 2 output repo path explicitly
#         repo_type="model"
#     )
#     print(f"🎉 Success! Stage 2 evaluation matrix deployed to: {OUTPUT_REPO_ID}/blob/main/{path_in_repo}")

# except Exception as e:
#     print(f"❌ Hugging Face push aborted. Error context details: {e}")

In [ ]:
import os, re
from huggingface_hub import HfApi, hf_hub_download

# 1. Define separate Hugging Face repositories for Input vs Output
INPUT_REPO_ID = "Pzazz55/insurance-ai-assistant-finetuning-stage1"
OUTPUT_REPO_ID = "Pzazz55/insurance-ai-assistant-finetuning-stage2"
# HF_TOKEN = "YOUR_HF_WRITE_TOKEN"

local_dir = "/content/reports"
os.makedirs(local_dir, exist_ok=True)
output_report_path = os.path.join(local_dir, "final_evaluation.md")

# import os
# import re
# from huggingface_hub import HfApi

# # 1. Define your Hugging Face configuration
# HF_REPO_ID = "Pzazz55/insurance-ai-assistant-finetuning-stage1"
# # HF_TOKEN = "YOUR_HF_WRITE_TOKEN"

input_report_path = "/content/reports/stage1_evaluation_report.md"
output_report_path = "/content/reports/stage2_evaluation_report.md"

# 2. Read and parse the existing stage1 markdown report
if not os.path.exists(input_report_path):
    raise FileNotFoundError(
        f"Could not find {input_report_path}. Please make sure Stage 1 script ran and generated it first!"
    )

print(f"📖 Reading existing report from {input_report_path}...")
with open(input_report_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Reconstruct the header and extract existing rows
markdown = []
existing_rows = []

for line in lines:
    line_str = line.strip()

    # Update the title header if found
    if line_str.startswith("# Stage 1 Evaluation Report"):
        markdown.append("# Stage 2 Evaluation Report")
    # Update the table column header to add the new column
    elif "| Question |" in line_str and "Model Answer" in line_str:
        markdown.append("| Question | Base Model Answer | Fine-Tuned Model Answer |")
    # Update the markdown table separator line
    elif "|---|---|" in line_str or "|---|---|---|" in line_str:
        markdown.append("|---|---|---|")
    # Parse existing table rows (skipping empty lines or headers)
    elif line_str.startswith("|") and not line_str.startswith("| Question"):
        # Split the row by '|', filter out empty spaces
        parts = [p.strip() for p in line_str.split("|")[1:-1]]
        if len(parts) >= 2:
            question = parts[0]
            base_answer = parts[1]
            existing_rows.append((question, base_answer))
    elif line_str != "":
        markdown.append(line_str)

# Ensure blank space after markdown meta header text
if markdown[-1] != "":
    markdown.append("")

print(f"📋 Found {len(existing_rows)} rows to update.")
print("🚀 Starting Fine-Tuned Evaluation Loop...")
print("-" * 50)

# 3. Loop through extracted questions to append the Fine-Tuned Answer
for idx, (question, base_answer) in enumerate(existing_rows, 1):
    # To pass clean text to the model, clean up any HTML markdown artifacts if they exist
    clean_question = question.replace("<br>", "\n").replace("<br/>", "\n")

    print(f"\n[Question {idx}/{len(existing_rows)}]: {clean_question}")
    print("-" * 50)

    # Generate the fine-tuned model's prediction
    ft_answer = generate_answer(clean_question)

    print(f"[Fine-Tuned Model Answer]:\n{ft_answer}")
    print("-" * 50)

    # Format line breaks cleanly for markdown columns
    md_safe_ft_answer = ft_answer.replace(chr(10), "<br>")

    # Append the row combining old columns with the new Fine-Tuned column
    markdown.append(f"| {question} | {base_answer} | {md_safe_ft_answer} |")

# 4. Save the final report locally
os.makedirs(os.path.dirname(output_report_path), exist_ok=True)
with open(output_report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(markdown))

print(f"\n✅ Evaluation complete! Local stage 2 report saved to {output_report_path}")
print("-" * 50)

# 5. Push the new file directly to your Hugging Face repository
print(f"📤 Uploading report to Hugging Face Hub ({HF_REPO_ID})...")
try:
    api = HfApi(token=HF_TOKEN)
    path_in_repo = "results/stage2_evaluation_report.md"

    api.upload_file(
        path_or_fileobj=output_report_path,
        path_in_repo=path_in_repo,
        repo_id=HF_REPO_ID,
        repo_type="model" # Change to "dataset" if your repository is a Dataset type instead of a Model type
    )
    print(f"🎉 Success! File saved in HF repo at: results/stage2_evaluation_report.md")

except Exception as e:
    print(f"❌ Hugging Face upload failed. Error details: {e}")

In [ ]:
import os
import re
import torch
from unsloth import FastLanguageModel
from safetensors.torch import load_file
from huggingface_hub import HfApi

# =====================================================================
# CONFIGURATION & SEPARATE REPOSITORY SETUP
# =====================================================================
max_seq_length = 1024
production_model_path = "/content/models/stage2_sft"

# Separate input/output configs aligned with your update
INPUT_REPO_ID = "Pzazz55/insurance-ai-assistant-finetuning-stage1"
OUTPUT_REPO_ID = "Pzazz55/insurance-ai-assistant-finetuning-stage2"

local_dir = "/content/reports"
os.makedirs(local_dir, exist_ok=True)

input_report_path = os.path.join(local_dir, "stage1_evaluation_report.md")
output_report_path = os.path.join(local_dir, "stage2_evaluation_report.md")

if not os.path.exists(production_model_path):
    raise FileNotFoundError(f"Could not find the model path at {production_model_path}. Ensure Stage 2 SFT completed successfully.")

if not os.path.exists(input_report_path):
    raise FileNotFoundError(
        f"Could not find {input_report_path}. Please make sure Stage 1 script ran and generated it first!"
    )

# =====================================================================
# 1. LOCAL PRODUCTION MODEL INITIALIZATION (DIRECT SAFETENSORS MAPPING)
# =====================================================================
print("🧠 Loading underlying base model architecture blueprint...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

print(f"🧬 Injecting custom Stage 2 SFT Safetensors weights from {production_model_path}...")
safetensors_weight_path = os.path.join(production_model_path, "model.safetensors")
state_dict = load_file(safetensors_weight_path, device="cuda")

# Map local fine-tuned weights matrix onto active architecture structural skeleton
model.load_state_dict(state_dict, strict=False)
FastLanguageModel.for_inference(model)
print("✅ SFT Production Model loaded successfully and ready for evaluation!\n")

# =====================================================================
# 2. READ AND PARSE THE EXISTING STAGE 1 MARKDOWN REPORT
# =====================================================================
print(f"📖 Reading existing report from {input_report_path}...")
with open(input_report_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

markdown = []
existing_rows = []

for line in lines:
    line_str = line.strip()

    # Update title header alignment
    if line_str.startswith("# Stage 1 Evaluation Report"):
        markdown.append("# Stage 2 Evaluation Report")
    # Update the layout headers to add the fine-tuned evaluation dimension
    elif "| Question |" in line_str and "Model Answer" in line_str:
        markdown.append("| Question | Base Model Answer | Fine-Tuned Model Answer |")
    # Balance table margins gracefully
    elif any(sep in line_str for sep in ["|---|---|", "|---|---|---|", "| :--- | :--- |"]):
        markdown.append("| :--- | :--- | :--- |")
    # Parse existing evaluation table rows
    elif line_str.startswith("|") and not line_str.startswith("| Question"):
        parts = [p.strip() for p in line_str.split("|")[1:-1]]
        if len(parts) >= 2:
            question = parts[0]
            base_answer = parts[1]
            existing_rows.append((question, base_answer))
    elif line_str != "":
        markdown.append(line_str)

if markdown[-1] != "":
    markdown.append("")

print(f"📋 Found {len(existing_rows)} rows to update.")
print("🚀 Starting Fine-Tuned Evaluation Loop using Token Slicing...")
print("-" * 50)

# =====================================================================
# 3. EVALUATION MATRIX GENERATION LOOP
# =====================================================================
for idx, (question, base_answer) in enumerate(existing_rows, 1):
    # Convert cell breaks back into plaintext formatting strings for inference execution
    clean_question = question.replace("<br>", "\n").replace("<br/>", "\n")

    print(f"\n[Question {idx}/{len(existing_rows)}]: {clean_question}")
    print("-" * 50)

    # Structure text inside aligned ChatML structure using chat templates
    messages = [
        {"role": "system", "content": "You are an expert Insurance Underwriting Assistant. Provide clear and legally accurate domain responses."},
        {"role": "user", "content": clean_question}
    ]
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([formatted_prompt], return_tensors = "pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = 300,
            use_cache = True,
            eos_token_id = tokenizer.eos_token_id,
            temperature = 0.1
        )

    # Slice output sequence to pull exactly the freshly minted inference tokens
    input_len = inputs.input_ids.shape[1]
    generated_tokens = outputs[0][input_len:]
    ft_answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    print(f"[Fine-Tuned Model Answer]:\n{ft_answer}")
    print("-" * 50)

    # Sanitize markdown strings against formatting breaks
    md_safe_ft_answer = ft_answer.replace("\n", "<br>").replace("|", "\\|")

    # Append the completed data string straight back to the dataset row stack
    markdown.append(f"| {question} | {base_answer} | {md_safe_ft_answer} |")

# =====================================================================
# 4. EXPORT COMPILED MARKDOWN REPORT TO LOCAL STORAGE
# =====================================================================
with open(output_report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(markdown))

print(f"\n✅ Evaluation complete! Local stage 2 report saved to {output_report_path}")
print("-" * 50)

# =====================================================================
# 5. STREAM REMOTE TRANSFER DIRECT TO STAGE 2 HUGGING FACE RESULTS FOLDER
# =====================================================================
print(f"📤 Uploading report to Hugging Face Hub under Stage 2 repository results folder...")
try:
    api = HfApi(token=HF_TOKEN)
    path_in_repo = "results/stage2_evaluation_report.md"

    api.upload_file(
        path_or_fileobj=output_report_path,
        path_in_repo=path_in_repo,
        repo_id=OUTPUT_REPO_ID,  # FIXED: Target the Stage 2 output repo path explicitly
        repo_type="model"
    )
    print(f"🎉 Success! Stage 2 evaluation matrix deployed to: {OUTPUT_REPO_ID}/blob/main/{path_in_repo}")

except Exception as e:
    print(f"❌ Hugging Face push aborted. Error context details: {e}")